# 09. Klaszter és Tipológia Elemzés

**Cél**: A kőbányai lakáspiaci kínálat automatikus tipológiájának és természetes alpiacainak feltárása nem felügyelt gépi tanulási módszerekkel (K-Means és Hierarchikus klaszterezés).

**Módszertan**: Robusztus standardizálás (`StandardScaler`), optimális klaszterszám meghatározása az Elbow (Inertia) módszerrel és Silhouette pontszámokkal, valamint dimenziócsökkentés Főkomponens-elemzéssel (PCA).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elemzett lakásállomány: {len(elado)} db.")

### 1. Klaszterezési Eredmények és Tipológiai KPI-k
K=4 szegmens illesztése a normalizált ingatlanjellemzőkre (fajlagos ár, alapterület, szobaszám, állapot, épületkora).

In [ ]:
cluster_vars = ['nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'allapot_kod', 'epulet_kora_ev']
df_km = elado.dropna(subset=cluster_vars).copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_km[cluster_vars])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_km['klaszter'] = kmeans.fit_predict(X_scaled)

# Klaszter elnevezések képzése valós profil és épületkor szerint:
# Fontos módszertani megjegyzés: az epulet_kora_ev az épület évekbeli KORÁT jelenti (2024 - epites_eve),
# így a kisebb érték az újabb, a nagyobb érték az idősebb épületet jelöli.
means = df_km.groupby('klaszter')[cluster_vars].mean()
cluster_names = {
    0: '1. Új építésű prémium kis lakások (átlagkor: ~5 év)',
    1: '2. Régebbi kompakt lakások (Panel/Tégla átlag, ~60 év)',
    2: '3. Idősebb nagyméretű lakások (Kedvező fajlagos ár, ~55 év)',
    3: '4. Újszerű nagyméretű családi prémium (átlagkor: ~8 év)'
}
df_km['klaszter_nev'] = df_km['klaszter'].map(cluster_names)

sil = silhouette_score(X_scaled, df_km['klaszter'])

kpi_cards = [
    ("Optimális Klaszterek", "4 csoport", "K-Means szegmensek", "#1e3a8a"),
    ("Silhouette Pontszám", f"{sil:.3f}", "Klaszter szeparáció jósága", "#059669"),
    ("1. Szegmens Méret", f"{(df_km['klaszter']==0).sum()} db", "Új prémium kis lakás", "#10b981"),
    ("2. Szegmens Méret", f"{(df_km['klaszter']==1).sum()} db", "Régebbi kompakt átlag", "#2563eb"),
    ("3. Szegmens Méret", f"{(df_km['klaszter']==2).sum()} db", "Idősebb nagylakás", "#d97706"),
    ("4. Szegmens Méret", f"{(df_km['klaszter']==3).sum()} db", "Újszerű nagy prémium", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Optimális Klaszterszám (Elbow Görbe és Silhouette)
Az inercia csökkenése és a Silhouette pontszámok alakulása K=2..8 klaszterszámra.

In [ ]:
ks = list(range(2, 9))
inertias = []
sils = []

for k in ks:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(X_scaled)
    inertias.append(km_temp.inertia_)
    sils.append(silhouette_score(X_scaled, km_temp.labels_))

fig1 = make_subplots(specs=[[{"secondary_y": True}]])
fig1.add_trace(
    go.Scatter(x=ks, y=inertias, mode='lines+markers', name='Inertia (Elbow görbe)', line=dict(color='#2563eb', width=3)),
    secondary_y=False
)
fig1.add_trace(
    go.Scatter(x=ks, y=sils, mode='lines+markers', name='Silhouette Score', line=dict(color='#10b981', width=3)),
    secondary_y=True
)
fig1.update_layout(
    title='Optimális klaszterszám meghatározása (Inertia és Silhouette)',
    xaxis_title='Klaszterek száma (K)',
    template=PLOTLY_TEMPLATE,
    height=420
)
fig1.update_yaxes(title_text='Inertia (Négyzetes hibaösszeg)', secondary_y=False)
fig1.update_yaxes(title_text='Silhouette Score', secondary_y=True)
fig1.show()

# 2D PCA Vetület
pca = PCA(n_components=2)
coords_pca = pca.fit_transform(X_scaled)
df_km['pca_x'] = coords_pca[:, 0]
df_km['pca_y'] = coords_pca[:, 1]

fig2 = px.scatter(
    df_km,
    x='pca_x',
    y='pca_y',
    color='klaszter_nev',
    title='Lakáspiaci szegmensek 2D PCA projekciója',
    labels={'pca_x': f'PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', 'pca_y': f'PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', 'klaszter_nev': 'Szegmens'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=480)
fig2.show()

### 3. Klaszterprofilok és Városrészi Összetétel
A négy lakástípus átlagos tulajdonságai és eloszlásuk a kőbányai városrészekben.

In [ ]:
# Összefoglaló statisztika táblázat
cluster_summary = df_km.groupby('klaszter_nev')[cluster_vars].mean().reset_index()
display(HTML("<b>Klaszterek átlagos jellemzői:</b><br>" + cluster_summary.round(1).to_html(classes='table table-bordered table-striped', index=False)))

# Standardizált radar ábra a klaszterprofilokhoz
fig_radar = go.Figure()
scaler_radar = StandardScaler()
df_radar_scaled = pd.DataFrame(scaler_radar.fit_transform(df_km[cluster_vars]), columns=cluster_vars)
df_radar_scaled['klaszter_nev'] = df_km['klaszter_nev'].values
radar_agg = df_radar_scaled.groupby('klaszter_nev')[cluster_vars].mean().reset_index()

for i, row in radar_agg.iterrows():
    fig_radar.add_trace(go.Scatterpolar(
        r=row[cluster_vars].values,
        theta=cluster_vars,
        fill='toself',
        name=row['klaszter_nev']
    ))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    showlegend=True,
    title='Standardizált Klaszterprofilok (Radar Diagram)',
    template=PLOTLY_TEMPLATE,
    height=500
)
fig_radar.show()

# Kereszttábla városrészek szerint
ct = pd.crosstab(df_km['varosresz'], df_km['klaszter_nev'])

fig3 = px.imshow(
    ct,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Városrészek és ingatlanpiaci klaszterek kereszttáblája (darabszám)',
    labels=dict(x="Ingatlan Szegmens", y="Városrész", color="Darabszám"),
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=450, xaxis_tickangle=-30)
fig3.show()

### 4. Interaktív Klaszterezési Sandbox
Tesztelje interaktívan a klaszterezést különböző klaszterszámok (K=2..6) beállításával!

In [ ]:
w_k_slider = widgets.IntSlider(min=2, max=6, value=4, description='Klaszter K:')
out_km = widgets.Output()

def frissit_km(*args):
    k = w_k_slider.value
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    s_score = silhouette_score(X_scaled, labels)
    
    with out_km:
        clear_output(wait=True)
        display(HTML(kpi_grid_html([
            ("Klaszterszám", f"{k} db", "Választott K", "#2563eb"),
            ("Silhouette Index", f"{s_score:.3f}", "Elkülönültség", "#10b981")
        ])))
        fig = px.scatter(
            x=coords_pca[:, 0], y=coords_pca[:, 1], color=[f'Klaszter {c+1}' for c in labels],
            title=f'K={k} Klaszter PCA vetülete (Silhouette={s_score:.3f})',
            template=PLOTLY_TEMPLATE
        )
        fig.show()

w_k_slider.observe(frissit_km, names='value')
display(w_k_slider)
display(out_km)
frissit_km()